# Actividad: Aritmética de Punto Flotante

## Parte 1: Preguntas conceptuales

Responda las siguientes preguntas en forma clara y concisa.

In [1]:
import numpy as np

### 1. **Representación binaria vs representación decimal:** 
Explique por qué la expresión `0.1 + 0.2 == 0.3` da el resultado `False` en Python. No responda simplemente “por errores de redondeo”; explique los detalles específicos sobre cómo 0.1 es representado en notación binaria comparada con su representación decimal.

**R.1**:\
Efectivamente, la expresión es falsa por el redondeo, sin embargo está causado por lo que vimos en clase de la notación IEEE754.

Básicamente, tanto el $0.1$ como el $0.2$ dan una fracción irracional en el cómputo a notación binaria, y al tener "sólo" 53 bits disponibles para almacenar la mantiza, se trunca la representación de ambos números.

Por lo tanto, al sumarlos no tenemos 0.3, si no su aproximación: por ende, la expresión `0.1 + 0.2 == 0.3` es computada como `False`


In [2]:
0.1 + 0.2 == 0.3

False


### 2. **Falta de asociatividad:** 
En matemáticas, la suma de números reales es asociativa, es decir, si $a, b, c$ son reales, tenemos que $(a+b) + c = a + (b + c)$. Explique por qué esta propiedad no siempre se cumple en aritmética de punto flotante. Dé un ejemplo cualitativo de un escenario en donde el orden de la adición podría cambiar el resultado significativamente.

**R.2**:\
La regla de asociatividad pierde sentido en la aritmética flotante pues el redondeo se hace por operación, entonces se acumula de forma distinta. Usando el ejemplo general para 3 números, 
$$(a+b) + c \neq a + (b + c)$$
Porque el redondeo de $(a + b)$ es diferente al redondeo de $(b + c)$, **por lo tanto las expresiones, aunque por poco, en términos de bits no son iguales.** 



In [3]:
(0.1 + 0.2) + 0.3 == 0.1 + (0.2 + 0.3)

False

### 3. **Cancelación catastrófica:**
Defina “Cancelación catastrófica” con sus propias palabras.  
¿Por qué este tipo de error en específico es considerado más peligroso que el error de redondeo estándar?

**R.3**:\
La cancelación catrastófica ocurre cuando restamos dos números muy similares binariamente de tal manera que los dígitos más representativos *"se cancelan"*, esto causa que la resta sea entre los dígitos con menor precisión, y por lo tanto el error sea impreciso numéricamente. 
Por eso es peor que el error de reondeo estándar, pues en ese el error es "proporcional" a la diferencia entre los dígitos, sin embargo en la cancelación catastrófica los resultados son disparates matemáticos.

In [4]:
a = 1.0000000000001
b = 1
print(f'a - b = {a - b}')

a - b = 9.992007221626409e-14


### 4. **Diferenciación numérica:** 
Cuando se calcula una derivada numéricamente usando la fórmula de diferencias finitas:
   $$f'(x) \approx \frac{f(x+h) - f(x)}{h}$$
   en principio se desea que $h$ sea tan pequeño como sea posible para minimizar el error de truncamiento. Sin embargo, si hacemos $h$ demasiado pequeño, por ejemplo, $10^{-15}$, el error en realidad crece drásticamente. ¿Por qué?

**R.4**:

El error es feo por la siguiente razón: al aproximar un número muy pequeño a $0$, la diferencia en el cómputo de $f(x-h)-f(x)$ es insignificante, y por lo tanto probable a cancelación catastrófica.\
Además, para empeorar las cosas, al dividir esa cantidad "alterada" por cancelación catastrófica por un número muy pequeño, estamos haciendo más grande el error por proporciones inversas al tamaño del divisor: es decir, estamos empeorando por mucho el error.\
Simplemente:
$$
\text{Si $h$ es muy pequeño:} \newline
\frac{1}{h}
\text{ Se vuelve muy grande.} \newline 
\text{por lo que al computar: }\newline f(x-h)-f(x)\newline 
\text{El error se acentúa mucho.}
$$


In [5]:
def f(x):
    return np.square(x)

def derivative_f(x, h):
    return (f(x-h) * f(x))/h

a = 1.0000001
h = 10e-5
print(f"Esta diferencia es de locos: {derivative_f(a, h):.4f}")

Esta diferencia es de locos: 9998.0041


### 5. **Raíz cuadrada inversa:**
 Busque información sobre el algoritmo para calcular el inverso de la raíz cuadrada de un número (*Fast Inverse Square Root*). Basándose en dicha información, escriba un breve texto donde incluya el algoritmo, una descripción de los pasos que aprovechan las características de números de punto flotante, y ejemplos de implementación en videojuegos "retro".

Esta fue la respuesta de la IA (se lo pedí en código Markdown):

**R.5**:\
Este legendario algoritmo se hizo famoso al descubrirse en el código fuente del motor gráfico de *Quake III Arena* (1999). Fue diseñado para resolver la ecuación $y = \frac{1}{\sqrt{x}}$, una operación matemática extremadamente costosa pero indispensable en el mundo de los gráficos 3D y los videojuegos para la normalización de vectores (usada intensivamente en cálculos de iluminación, sombreado y reflejos).

El algoritmo es una obra maestra de la optimización que aprovecha la forma en que el estándar IEEE 754 almacena los números flotantes:

1.  **Aliasing de bits:** Toma los bits que conforman el número de punto flotante ($x$) y los interpreta directamente como si fueran un número entero (`long`).
2.  **La magia constante:** Realiza un desplazamiento de bits hacia la derecha (`i >> 1`), lo que en enteros equivale a dividir por 2 (y matemáticamente aproxima la raíz cuadrada de un exponente). Luego, resta ese resultado de un número mágico hexadecimal: `0x5f3759df`. De forma fascinante, esta resta de enteros entrega una aproximación sorprendentemente buena de la raíz cuadrada inversa en formato flotante.
3.  **Conversión inversa:** Vuelve a interpretar los bits de ese número entero resultante como un número de punto flotante ($y$).
4.  **Refinamiento (Newton-Raphson):** La aproximación anterior tiene un pequeño margen de error. Para corregirlo rápidamente, el código ejecuta una única iteración del método de Newton-Raphson: `y = y * (1.5 - (x2 * y * y))`, lo cual pule el resultado para uso en el juego.

En el hardware de los años 90, evitar la división flotante y calcular esto manipulando bits a nivel de enteros permitía que el cálculo fuera hasta cuatro veces más rápido, lo que significaba la diferencia entre un juego con gráficos entrecortados y uno de rendimiento fluido.


## Parte 2: Ejercicios computacionales

Use el archivo `1 - Floating point arithmetic.py` para implementar sus soluciones.



### 1: La serie armónica

La serie armónica (finita) está definida por:
$$\sum_{n=1}^{N} \frac{1}{n}$$
Dado que la precisión de punto flotante es finita, el orden en que se suman los términos de la serie es importante.

* Implemente la función `harmonic_sum_forward(n)`. Esta función debe calcular la suma de la serie armónica de 1 a N (en ese orden).
* Implemente la función `harmonic_sum_backward(n)`. Esta función debe calcular la suma de la serie armónica de N a 1 (es decir, en orden inverso).
* Ejecute ambas funciones para N = 1,000,000.
* **Análisis:** ¿Cuál método es más preciso y por qué? El valor exacto de la serie del punto anterior, con 32 cifras significativas, es 14.392726722865723631381127493189.


**R.6**:\
Sumar al revés es más preciso. Esto es debido a que sumando de esa manera comenzamos con los números pequeños, y el error que vamos acumulando en cuestiones de bits es menor, pues se "acumulan" en los números pequeños, y al momento de sumar con los números grandes posteriormente, los bits menos significatovos se conservan y minimizan el error por redondeo. Por otro lado, si empezaremos sumando como la sucesión lo indica, el total se vuelve muy grande comparado con los números que estamos sumando, y como sabemos, si sumamos a un número grande un número muy pequeño, los bits menos significativo del número pequeño se truncan o se pierden por cancelación catastrófica.\
**Por lo tanto, la suma al revés es más precisa.** 


In [ ]:
# Ran with the terminal:
"""
--- Task 1: Harmonic Series (N=1,000,000) ---
Forward Sum:  14.39272672286498888639
Backward Sum: 14.39272672286577225975
Difference:   0.00000000000078337337

"""


### 2: Cálculo estable de la varianza

La fórmula estándar para calcular la varianza de un conjunto de datos $X = \{x_1, x_2, ..., x_N\}$ es:
$$Var(X) = \frac{1}{N} \sum_{i=1}^{N} (x_i - \bar{x})^2$$

Con frecuencia, en los libros de estadística se ofrece una fórmula alternativa “de un paso” que es más barata computacionalmente:
$$Var(X) = \frac{1}{N} \sum_{i=1}^{N} x_i^2 - \bar{x}^2$$

Sin embargo, la segunda fórmula es numéricamente inestable cuando la varianza es pequeña comparada con la magnitud de los datos (por ejemplo, si $X = \{1000000.1, 10000000.2, ...\}$).

* Implemente la función `variance_naive(data)` usando la fórmula inestable de un paso.
* Implemente la función `variance_stable(data)` usando el algoritmo de dos pasos (calcular la media aritmética primero y luego iterar la suma de diferencias cuadradas) o el algoritmo de Welford.
* Pruebe ambas funciones con el conjunto de datos dado en el archivo adjunto.


In [ ]:
"""
--- Task 2: Variance Calculation ---
Naive Variance:  128.0
Stable Variance: 0.006666661898303043

"""



### 3: La fórmula cuadrática revisitada

En clase usted aprendió que la fórmula cuadrática estándar falla cuando $b^2 \approx 4ac$.

* Implemente la función `solve_quadratic(a, b, c)` que devuelve una tupla con las raíces `(x1, x2)`.
* Su función implementada debe detectar cuál raíz causa cancelación catastrófica (basada en el signo de $b$) y use la fórmula cuadrática alterna para esa raíz en específico.
* Pruebe su función con a = 1, b = $10^8$, c = 1.

In [ ]:
"""
--- Task 3: Quadratic Formula ---
Roots for a=1.0, b=100000000, c=1.0:
Root 1: -100000000.0
Root 2: -1e-08

"""